# Governance for Claude Tool-Use Agents

This cookbook demonstrates how to add deterministic governance to Claude agents that use tools (function calling). You'll learn how to:

1. **Scan tool arguments for PII** before executing tools
2. **Track and cap costs** across multi-turn agent conversations
3. **Enforce tool allowlists** — restrict which tools the agent can call
4. **Produce structured audit evidence** for every tool call decision

All governance runs deterministically in <5ms with no additional LLM calls.

## Why governance for tool-use?

When Claude decides to call a tool, the arguments are generated by the model. In production, this means:
- PII from user messages can leak into tool arguments (e.g., SSNs passed to a database query)
- Runaway tool loops can burn budget without a hard stop
- Compliance teams need evidence that every tool call was evaluated

This notebook shows a pattern for intercepting `tool_use` blocks before execution.

## Setup

In [ ]:
%pip install anthropic --quiet

In [ ]:
import os
import re
import json
import time
import uuid
from dataclasses import dataclass, field
from typing import Any

import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Step 1: Define a Governance Engine

This lightweight engine evaluates deterministic policies against tool call arguments. No LLM in the governance path — just regex patterns and allowlists.

In [ ]:
# PII detection patterns
PII_PATTERNS = {
    "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "credit_card": re.compile(r"\b(?:\d{4}[-\s]?){3}\d{4}\b"),
    "email": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"),
}

SECRET_PATTERNS = [
    re.compile(r"(?i)(?:api[_-]?key|apikey)\s*[:=]\s*['\"]?[a-zA-Z0-9_\-]{20,}"),
    re.compile(r"(?:AKIA|ASIA)[A-Z0-9]{16}"),
    re.compile(r"-----BEGIN (?:RSA |EC |DSA )?PRIVATE KEY-----"),
]


@dataclass
class GovernanceDecision:
    """Structured record of a governance evaluation."""
    decision_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    action: str = "ALLOW"  # ALLOW or DENY
    tool_name: str = ""
    reason: str = ""
    reason_codes: list = field(default_factory=list)
    evaluation_time_ms: float = 0.0
    timestamp: float = field(default_factory=time.time)


class GovernanceEngine:
    """Deterministic policy engine for tool-use governance."""

    def __init__(
        self,
        allowed_tools: list[str] | None = None,
        pii_categories: list[str] | None = None,
        budget_limit: float = float("inf"),
        cost_per_call: float = 0.002,
    ):
        self.allowed_tools = allowed_tools  # None = allow all
        self.pii_categories = pii_categories or ["ssn", "credit_card"]
        self.budget_limit = budget_limit
        self.cost_per_call = cost_per_call
        self.cumulative_cost = 0.0
        self.decisions: list[GovernanceDecision] = []

    def evaluate(self, tool_name: str, arguments: dict[str, Any]) -> GovernanceDecision:
        """Evaluate a tool call against all policies. Returns ALLOW or DENY."""
        start = time.perf_counter()

        # 1. Tool allowlist
        if self.allowed_tools is not None and tool_name not in self.allowed_tools:
            decision = GovernanceDecision(
                action="DENY",
                tool_name=tool_name,
                reason=f"Tool '{tool_name}' not in allowlist",
                reason_codes=["TOOL_NOT_ALLOWED"],
                evaluation_time_ms=(time.perf_counter() - start) * 1000,
            )
            self.decisions.append(decision)
            return decision

        # 2. Budget check
        if self.cumulative_cost >= self.budget_limit:
            decision = GovernanceDecision(
                action="DENY",
                tool_name=tool_name,
                reason=f"Budget exceeded: ${self.cumulative_cost:.4f} >= ${self.budget_limit:.2f}",
                reason_codes=["BUDGET_EXCEEDED"],
                evaluation_time_ms=(time.perf_counter() - start) * 1000,
            )
            self.decisions.append(decision)
            return decision

        # 3. PII scan
        args_str = json.dumps(arguments)
        for category in self.pii_categories:
            pattern = PII_PATTERNS.get(category)
            if pattern and pattern.search(args_str):
                decision = GovernanceDecision(
                    action="DENY",
                    tool_name=tool_name,
                    reason=f"PII detected in arguments: {category}",
                    reason_codes=[f"PII_DETECTED:{category}"],
                    evaluation_time_ms=(time.perf_counter() - start) * 1000,
                )
                self.decisions.append(decision)
                return decision

        # 4. Secret scan
        for pattern in SECRET_PATTERNS:
            if pattern.search(args_str):
                decision = GovernanceDecision(
                    action="DENY",
                    tool_name=tool_name,
                    reason="Secret detected in arguments",
                    reason_codes=["SECRET_DETECTED"],
                    evaluation_time_ms=(time.perf_counter() - start) * 1000,
                )
                self.decisions.append(decision)
                return decision

        # All checks passed
        self.cumulative_cost += self.cost_per_call
        decision = GovernanceDecision(
            action="ALLOW",
            tool_name=tool_name,
            reason="All governance checks passed",
            evaluation_time_ms=(time.perf_counter() - start) * 1000,
        )
        self.decisions.append(decision)
        return decision


print("GovernanceEngine ready.")

## Step 2: Define Tools and a Governed Execution Loop

We define tools that Claude can call, then wrap the execution with governance checks.

In [ ]:
# Tool definitions for Claude
tools = [
    {
        "name": "search_customers",
        "description": "Search customer records by name or ID",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
            },
            "required": ["query"],
        },
    },
    {
        "name": "send_email",
        "description": "Send an email to a customer",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email"},
                "subject": {"type": "string"},
                "body": {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
    {
        "name": "delete_account",
        "description": "Delete a customer account permanently",
        "input_schema": {
            "type": "object",
            "properties": {
                "account_id": {"type": "string"},
            },
            "required": ["account_id"],
        },
    },
]


def execute_tool(tool_name: str, arguments: dict) -> str:
    """Simulate tool execution (in production this calls real services)."""
    if tool_name == "search_customers":
        return json.dumps({"results": [{"name": "Alice Smith", "id": "C-1234"}]})
    elif tool_name == "send_email":
        return json.dumps({"status": "sent", "message_id": "msg-abc"})
    elif tool_name == "delete_account":
        return json.dumps({"status": "deleted"})
    return json.dumps({"error": "unknown tool"})


print(f"Defined {len(tools)} tools: {[t['name'] for t in tools]}")

## Step 3: Run the Agent with Governance

The key pattern: after Claude responds with `tool_use` blocks, evaluate each one against the governance engine **before** executing. If denied, return a governance error to Claude instead of executing the tool.

In [ ]:
# Initialize governance: only allow search and email, cap budget at $0.01
engine = GovernanceEngine(
    allowed_tools=["search_customers", "send_email"],
    pii_categories=["ssn", "credit_card", "email"],
    budget_limit=0.01,
    cost_per_call=0.003,
)


def run_governed_agent(user_message: str, max_turns: int = 5):
    """Run a Claude agent with governance checks on every tool call."""
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # Check if Claude wants to use tools
        if response.stop_reason != "tool_use":
            # Final text response
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"\n🤖 Claude: {block.text}")
            break

        # Process each tool_use block with governance
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            tool_name = block.name
            arguments = block.input

            # === GOVERNANCE CHECK ===
            decision = engine.evaluate(tool_name, arguments)
            print(f"  📋 [{decision.action}] {tool_name}({json.dumps(arguments)[:60]}) — {decision.reason} ({decision.evaluation_time_ms:.2f}ms)")

            if decision.action == "DENY":
                # Return governance error to Claude (it can adapt)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": f"GOVERNANCE DENIED: {decision.reason}",
                    "is_error": True,
                })
            else:
                # Execute the tool
                result = execute_tool(tool_name, arguments)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        # Continue the conversation with tool results
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    return engine.decisions


print("Governed agent ready. Running...\n")
print("="*60)
print("Test 1: Normal search (should ALLOW)")
print("="*60)
decisions = run_governed_agent("Search for customer Alice Smith")

In [ ]:
# Reset engine for next test
engine = GovernanceEngine(
    allowed_tools=["search_customers", "send_email"],
    pii_categories=["ssn", "credit_card"],
    budget_limit=0.01,
)

print("="*60)
print("Test 2: Blocked tool (delete_account not in allowlist)")
print("="*60)
decisions = run_governed_agent("Delete account C-1234")

In [ ]:
# Reset engine for PII test
engine = GovernanceEngine(
    allowed_tools=["search_customers", "send_email"],
    pii_categories=["ssn", "credit_card"],
    budget_limit=0.01,
)

print("="*60)
print("Test 3: PII in tool arguments (SSN in search query)")
print("="*60)
decisions = run_governed_agent(
    "Search for the customer whose SSN is 123-45-6789"
)

## Step 4: Inspect the Audit Trail

Every governance decision is recorded with a unique ID, timestamp, evaluation time, and reason codes. This is the structured evidence compliance teams need.

In [ ]:
print(f"\n{'='*60}")
print(f"Audit Trail ({len(engine.decisions)} decisions)")
print(f"{'='*60}")

for d in engine.decisions:
    print(f"\n  ID: {d.decision_id[:8]}...")
    print(f"  Action: {d.action}")
    print(f"  Tool: {d.tool_name}")
    print(f"  Reason: {d.reason}")
    print(f"  Codes: {d.reason_codes}")
    print(f"  Latency: {d.evaluation_time_ms:.3f}ms")

## Summary

This pattern gives you:

| Capability | How |
|---|---|
| Tool allowlisting | Check tool name against a set before execution |
| PII detection | Regex scan of serialized arguments |
| Secret detection | Pattern matching for API keys, private keys |
| Cost governance | Cumulative tracking with hard stop |
| Audit trail | Structured decision records with IDs and timestamps |

All governance runs in-process, deterministically, in <1ms per evaluation.

### Production considerations

- Add more PII patterns (phone, address, passport) for your jurisdiction
- Use actual token costs from `response.usage` instead of flat per-call estimates
- Export audit records to your SIEM (Splunk, Datadog, etc.)
- Consider MONITOR mode (log but don't block) during rollout
- For a full implementation with 500+ detection patterns, see [TealTiger](https://github.com/agentguard-ai/tealtiger) (Apache 2.0)